In [0]:
%pip install tensorflow 

In [0]:

# Libraries Import


import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score
from pyspark.ml.functions import vector_to_array
import mlflow
from sklearn.preprocessing import StandardScaler

# Here the funciton to extract and normalize features is created. This function will be used later to extract the features from the training and test datasets.

def extract_and_normalize(train_df, test_df):
    """Extract arrays and normalize."""
    
    train_arr = train_df.withColumn("features_arr", vector_to_array("features"))
    test_arr  = test_df.withColumn("features_arr", vector_to_array("features"))

    X_train = np.array(train_arr.select("features_arr").toPandas()["features_arr"].tolist())
    y_train = train_arr.select("is_fraud").toPandas()["is_fraud"].values
    X_test  = np.array(test_arr.select("features_arr").toPandas()["features_arr"].tolist())
    y_test  = test_arr.select("is_fraud").toPandas()["is_fraud"].values

    # Only legit transactions for autoencoder training
    legit_mask = y_train == 0
    scaler = StandardScaler()
    scaler.fit(X_train[legit_mask])

    X_train_scaled = scaler.transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    print(f"  Feature means (should be ~0): {X_train_scaled[legit_mask].mean(axis=0).round(2)}")
    print(f"  Feature stds  (should be ~1): {X_train_scaled[legit_mask].std(axis=0).round(2)}")

    return X_train_scaled, y_train, X_test_scaled, y_test, scaler

# function is applied to both datasets
X_app_train, y_app_train, X_app_test, y_app_test, scaler_app = \
    extract_and_normalize(
        spark.table("workspace.ml_layer.application_train_features"),
        spark.table("workspace.ml_layer.application_test_features")
    )

X_txn_train, y_txn_train, X_txn_test, y_txn_test, scaler_txn = \
    extract_and_normalize(
        spark.table("workspace.ml_layer.transaction_train_features"),
        spark.table("workspace.ml_layer.transaction_test_features")
    )

# Legit-only training sets for autoencoder
X_app_legit = X_app_train[y_app_train == 0]
X_txn_legit = X_txn_train[y_txn_train == 0]

# Define number of features for each dataset
n_app_features = X_app_train.shape[1]
n_txn_features = X_txn_train.shape[1]


# Function for autoencoder

def build_autoencoder(n_features, name="autoencoder"):
    """
    Adaptive autoencoder architecture based on input dimensionality.
    Layer sizes are computed as proportions of n_features to ensure proper funneling.
    
    Architecture rule: every encoder layer must be smaller than the previous one.
    Bottleneck = ~30% of input size (sweet spot for tabular fraud detection).
    
    """
    # Bottleneck: ~30% of input, minimum 2
    encoding_dim = max(2, int(round(n_features * 0.3)))
    
    # Hidden layers: smooth funnel from input to bottleneck
    hidden_1 = max(encoding_dim + 2, int(round(n_features * 0.8)))   # ~80% of input
    hidden_2 = max(encoding_dim + 1, int(round(n_features * 0.5)))   # ~50% of input

    print(f"  Architecture for {n_features} features:")
    print(f"    {n_features} → {hidden_1} → {hidden_2} → {encoding_dim} → "
          f"{hidden_2} → {hidden_1} → {n_features}")
    print(f"    Bottleneck: {encoding_dim} ({100*encoding_dim/n_features:.0f}% of input)")

    # Adaptive dropout
    dropout_rate = 0.1 if n_features < 8 else 0.2

    # Adaptive learning rate
    learning_rate = 0.0005 if n_features < 8 else 0.001

    # Model building 
    inputs = keras.Input(shape=(n_features,), name="input")

    # Encoder
    x = layers.Dense(hidden_1, activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(hidden_2, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    encoded = layers.Dense(encoding_dim, activation="relu", name="bottleneck")(x)

    # Decoder
    x = layers.Dense(hidden_2, activation="relu")(encoded)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(hidden_1, activation="relu")(x)
    decoded = layers.Dense(n_features, activation="linear", name="output")(x)

    autoencoder = keras.Model(inputs, decoded, name=name)
    autoencoder.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse"
    )
    return autoencoder, encoding_dim


def run_autoencoder(X_legit_train, X_test, y_test, n_features, dataset_name, scaler):
    autoencoder, encoding_dim = build_autoencoder(n_features)   
    autoencoder.summary()

    # Validation split
    val_size = int(0.1 * len(X_legit_train))
    X_val    = X_legit_train[:val_size]
    X_tr     = X_legit_train[val_size:]

    with mlflow.start_run(run_name=f"Autoencoder_{dataset_name}"):
        history = autoencoder.fit(
            X_tr, X_tr,
            epochs=30,
            batch_size=512,                     # larger batch for stability
            validation_data=(X_val, X_val),     # legit-only validation
            shuffle=True,
            callbacks=[
                keras.callbacks.EarlyStopping(
                    monitor="val_loss",
                    patience=5,
                    restore_best_weights=True
                )
            ],
            verbose=1
        )

        # Reconstruction error
        X_reconstructed      = autoencoder.predict(X_test)
        reconstruction_error = np.mean(np.power(X_test - X_reconstructed, 2), axis=1)

        roc_auc = roc_auc_score(y_test, reconstruction_error)
        pr_auc  = average_precision_score(y_test, reconstruction_error)

        print(f"\n{'='*50}")
        print(f"  Autoencoder | {dataset_name}")
        print(f"{'='*50}")
        print(f"  ROC-AUC  : {roc_auc:.4f}")
        print(f"  PR-AUC   : {pr_auc:.4f}   ← key metric")

        threshold    = np.percentile(reconstruction_error, 95)
        y_pred       = (reconstruction_error >= threshold).astype(int)
        fraud_caught = y_test[y_pred == 1].sum()
        print(f"  Threshold (95th pct): {threshold:.4f}")
        print(f"  Fraud caught at threshold: {fraud_caught} / {y_test.sum()}")

        mlflow.log_metrics({"roc_auc": roc_auc, "pr_auc": pr_auc, "anomaly_threshold_95": threshold})
        mlflow.log_param("dataset", dataset_name)
        mlflow.log_param("n_features", n_features)
        mlflow.log_param("encoding_dim", encoding_dim)        
        mlflow.log_param("bottleneck_ratio", round(encoding_dim/n_features, 2))
        mlflow.keras.log_model(autoencoder, f"Autoencoder_{dataset_name}")
        mlflow.sklearn.log_model(scaler, f"Scaler_{dataset_name}")

    return autoencoder, reconstruction_error

# Function for Isolation Forest

def run_isolation_forest(X_train, X_test, y_test, dataset_name, contamination=0.01):
    iso = IsolationForest(
        n_estimators=200,
        contamination=contamination,   
        random_state=42,
        n_jobs=-1
    )

    with mlflow.start_run(run_name=f"IsolationForest_{dataset_name}"):
        iso.fit(X_train)

        # decision_function: lower score = more anomalous
        # Negative symbol before iso.decision_function() allow to align results to calculate roc_auc/pr_auc 
        anomaly_scores = -iso.decision_function(X_test)

        roc_auc = roc_auc_score(y_test, anomaly_scores)
        pr_auc  = average_precision_score(y_test, anomaly_scores)

        print(f"\n{'='*50}")
        print(f"  Isolation Forest | {dataset_name}")
        print(f"{'='*50}")
        print(f"  ROC-AUC : {roc_auc:.4f}")
        print(f"  PR-AUC  : {pr_auc:.4f}   ← key metric")

        mlflow.log_metrics({"roc_auc": roc_auc, "pr_auc": pr_auc})
        mlflow.log_param("dataset", dataset_name)
        mlflow.log_param("contamination", contamination)
        mlflow.sklearn.log_model(iso, f"IsoForest_{dataset_name}")

    return iso, anomaly_scores



# Run models
username = spark.sql("SELECT current_user()").collect()[0][0]

mlflow.set_experiment(f"/Users/{username}/anomaly_detection")
print(f"MLflow experiment: /Users/{username}/anomaly_detection")

# Applications
ae_app, ae_app_scores  = run_autoencoder(
    X_app_legit, X_app_test, y_app_test, n_app_features, "applications", scaler_app
)
iso_app, iso_app_scores = run_isolation_forest(
    X_app_train, X_app_test, y_app_test, "applications", contamination=0.01
)

# Transactions
ae_txn, ae_txn_scores  = run_autoencoder(
    X_txn_legit, X_txn_test, y_txn_test, n_txn_features, "transactions", scaler_txn
)
iso_txn, iso_txn_scores = run_isolation_forest(
    X_txn_train, X_txn_test, y_txn_test, "transactions", contamination=0.015
)